In [41]:
# Define some variables here
_NYT_CONNECTION_FILE = "/Users/yt/Desktop/nyt-connections-rag/data/ConnectionsFinalDataset.json"
#_NYT_CONNECTION_FILE = '/home/vgupta8/cs429/nyt-connections-rag/data/ConnectionsFinalDataset.json'
_DIFFICULTY = 4
_GAME_DATE = ""
_NUMBER_OF_CONNECTIONS = 4
#_GITHUB_TOKEN = "GITHUB_TOKEN"
#_URL =https://models.inference.ai.azure.com
_GitHUB_TOKEN = "OPENROUTER_API_KEY"
_URL ="https://openrouter.ai/api/v1"

In [4]:
# Helper functions for wordnet, evaluating connections, and other related tasks

from nltk.corpus import wordnet as wn

def get_word_synsets(words):
    """Return a dict mapping each word to its unique lemma names from WordNet."""
    result = {}
    for word in words:
        synsets = wn.synsets(word.lower().replace(" ", "_"))
        result[word] = list(dict.fromkeys(
            l.name().replace("_", " ") for s in synsets for l in s.lemmas()
        ))
    return result

## Evaluation metrics for predicted groups vs actual answers

def game_completion_rate(all_predicted, all_actual):
    """All 4 groups exactly correct → puzzle is 'completed'."""
    if not all_predicted:
        return 0.0
    completed = 0
    for predicted, actual in zip(all_predicted, all_actual):
        actual_sets = [set(a["words"]) for a in actual]
        pred_sets   = [set(p["words"]) for p in predicted]
        if all(ps in actual_sets for ps in pred_sets):
            completed += 1
    return completed / len(all_predicted)


def overall_accuracy(all_predicted, all_actual):
    """Proportion of predicted groups that exactly match a ground-truth group."""
    total, correct = 0, 0
    for predicted, actual in zip(all_predicted, all_actual):
        actual_sets = [set(a["words"]) for a in actual]
        for pred in predicted:
            total += 1
            if set(pred["words"]) in actual_sets:
                correct += 1
    return correct / total if total else 0.0


def partial_accuracy_rate(all_predicted, all_actual):
    """
    Proportion of predicted groups whose best overlap with any ground-truth group
    is exactly 3 or exactly 2 words (i.e. near-correct but not exact).
    Returns dict: {3: rate_for_3_overlap, 2: rate_for_2_overlap}
    """
    total = 0
    counts = {2: 0, 3: 0}
    for predicted, actual in zip(all_predicted, all_actual):
        actual_sets = [set(a["words"]) for a in actual]
        for pred in predicted:
            pred_set = set(pred["words"])
            best_overlap = max(len(pred_set & a_set) for a_set in actual_sets)
            total += 1
            if best_overlap in counts:
                counts[best_overlap] += 1
    return {k: v / total if total else 0.0 for k, v in counts.items()}



## Evaluation helpers — run once, then call evaluate_run() in the cells below

from IPython.display import display, HTML

def _badge(text, bg, fg="#fff"):
    return (f'<span style="background:{bg};color:{fg};padding:2px 8px;'
            f'border-radius:4px;font-size:0.85em;font-weight:bold">{text}</span>')

def render_puzzle(idx, total, contest, raw_reply, predicted, actual, scores):
    correct_groups = scores["correct_groups"]
    completed      = scores["completed"]
    group_acc      = scores["group_acc"]
    actual_sets    = [set(a["words"]) for a in actual]

    status_badge = (_badge("✓ SOLVED", "#2e7d32") if completed
                    else _badge(f"✗ {correct_groups}/4", "#c62828"))

    html = []
    html.append('<div style="border:1px solid #555;border-radius:8px;padding:16px;'
                'margin:12px 0;font-family:monospace">')

    # Header
    html.append('<div style="display:flex;align-items:center;gap:12px;margin-bottom:12px">')
    html.append(f'<b style="font-size:1.1em">[{idx}/{total}] {contest}</b>')
    html.append(status_badge)
    html.append(f'<span style="color:#888;font-size:0.85em">'
                f'acc {group_acc:.0%} | partial-3 {scores["partial_3"]:.0%} | partial-2 {scores["partial_2"]:.0%}'
                f'</span>')
    html.append('</div>')

    # Raw reply (collapsible)
    escaped = raw_reply.replace("&","&amp;").replace("<","&lt;").replace(">","&gt;")
    html.append('<details style="margin-bottom:12px">')
    html.append('<summary style="cursor:pointer;color:#90caf9">▶ Raw model reply</summary>')
    html.append(f'<pre style="background:#1e1e1e;color:#d4d4d4;padding:10px;border-radius:6px;'
                f'overflow-x:auto;white-space:pre-wrap;font-size:0.85em;margin-top:6px">{escaped}</pre>')
    html.append('</details>')

    # Side-by-side table
    html.append('<table style="width:100%;border-collapse:collapse;font-size:0.9em">')
    html.append('<tr>'
                '<th style="text-align:left;padding:4px 8px;border-bottom:1px solid #444;width:50%;color:#aaa">Predicted</th>'
                '<th style="text-align:left;padding:4px 8px;border-bottom:1px solid #444;width:50%;color:#aaa">Actual</th>'
                '</tr>')

    for row in range(max(len(predicted), len(actual))):
        html.append('<tr style="vertical-align:top">')

        if row < len(predicted):
            pred = predicted[row]
            ps = set(pred["words"])
            is_correct = ps in actual_sets
            bg = "#1b3a1b" if is_correct else "#3a1a1a"
            icon_color = "#66bb6a" if is_correct else "#ef5350"
            icon = "✓" if is_correct else "✗"
            label = pred.get("answerDescription", "?")
            if is_correct:
                words_display = [f'<span style="color:#66bb6a">{w}</span>' for w in pred["words"]]
                note = ""
            else:
                best_a = max(actual, key=lambda a: len(ps & set(a["words"])))
                best_set = set(best_a["words"])
                words_display = [
                    f'<span style="color:{"#66bb6a" if w in best_set else "#ef5350"}">{w}</span>'
                    for w in pred["words"]
                ]
                overlap = len(ps & best_set)
                note = (f'<div style="color:#888;font-size:0.8em;margin-top:2px">'
                        f'best match: <i>{best_a["answerDescription"]}</i> ({overlap}/4)</div>')
            html.append(
                f'<td style="padding:5px 8px;background:{bg};border-radius:4px">'
                f'<span style="color:{icon_color}">{icon}</span> '
                f'<b style="color:#ccc">{label}</b><br>'
                f'{", ".join(words_display)}{note}</td>'
            )
        else:
            html.append('<td></td>')

        if row < len(actual):
            a = actual[row]
            diff_colors = {"Yellow": "#f9a825", "Green": "#2e7d32",
                           "Blue": "#1565c0", "Purple": "#6a1b9a"}
            cat_color = diff_colors.get(a.get("difficulty", ""), "#555")
            html.append(
                f'<td style="padding:5px 8px;background:#1e1e2e;border-radius:4px">'
                f'<b style="color:{cat_color}">{a["answerDescription"]}</b><br>'
                f'<span style="color:#ccc">{", ".join(a["words"])}</span></td>'
            )
        else:
            html.append('<td></td>')

        html.append('</tr>')

    html.append('</table></div>')
    return "".join(html)


def evaluate_run( raw_replies, all_actual):
    """Evaluate a list of raw model replies against ground-truth answers. Returns per_puzzle_scores."""
    display(HTML(
        f'<h2 style="font-family:monospace;border-bottom:2px solid #555;padding-bottom:6px">'
    ))

    predicted_all  = []
    per_puzzle_scores = []

    for i, (raw_reply, actual) in enumerate(zip(raw_replies, all_actual)):
        try:
            s = raw_reply.find('[')
            e = raw_reply.rfind(']') + 1
            predicted = json.loads(raw_reply[s:e])
        except Exception as ex:
            print(f"[{i+1}] Failed to parse JSON: {ex}")
            predicted = []
        predicted_all.append(predicted)

        actual_sets = [set(a["words"]) for a in actual]
        pred_sets   = [set(p["words"]) for p in predicted]

        correct_groups = sum(1 for ps in pred_sets if ps in actual_sets)
        completed      = int(all(ps in actual_sets for ps in pred_sets) and len(pred_sets) == 4)
        partial_3 = sum(
            1 for ps in pred_sets
            if max((len(ps & a) for a in actual_sets), default=0) >= 3
        ) / max(len(pred_sets), 1)
        partial_2 = sum(
            1 for ps in pred_sets
            if max((len(ps & a) for a in actual_sets), default=0) >= 2
        ) / max(len(pred_sets), 1)
        group_acc = correct_groups / max(len(pred_sets), 1)

        scores = dict(correct_groups=correct_groups, completed=completed,
                      group_acc=group_acc, partial_3=partial_3, partial_2=partial_2)
        per_puzzle_scores.append({"contest": pool[i]["contest"], **scores})
        display(HTML(render_puzzle(i + 1, len(pool), pool[i]["contest"],
                                   raw_reply, predicted, actual, scores)))

    # Summary table
    n = len(per_puzzle_scores)
    avg_gcr = sum(s["completed"]  for s in per_puzzle_scores) / n
    avg_acc = sum(s["group_acc"]  for s in per_puzzle_scores) / n
    avg_p3  = sum(s["partial_3"]  for s in per_puzzle_scores) / n
    avg_p2  = sum(s["partial_2"]  for s in per_puzzle_scores) / n

    header = (
        '<tr style="color:#aaa;border-bottom:1px solid #555">'
        '<th style="padding:4px 12px;text-align:left">Contest</th>'
        '<th style="padding:4px 12px">Solved</th>'
        '<th style="padding:4px 12px">Group Acc</th>'
        '<th style="padding:4px 12px">Partial-3</th>'
        '<th style="padding:4px 12px">Partial-2</th></tr>'
    )
    rows = "".join(
        f'<tr><td style="padding:4px 12px">{s["contest"]}</td>'
        f'<td style="padding:4px 12px;text-align:center">{"✓" if s["completed"] else "✗"}</td>'
        f'<td style="padding:4px 12px;text-align:center">{s["group_acc"]:.0%}</td>'
        f'<td style="padding:4px 12px;text-align:center">{s["partial_3"]:.0%}</td>'
        f'<td style="padding:4px 12px;text-align:center">{s["partial_2"]:.0%}</td></tr>'
        for s in per_puzzle_scores
    )
    avg_row = (
        f'<tr style="border-top:2px solid #666;font-weight:bold">'
        f'<td style="padding:6px 12px">Average ({n})</td>'
        f'<td style="padding:6px 12px;text-align:center">{avg_gcr:.1%}</td>'
        f'<td style="padding:6px 12px;text-align:center">{avg_acc:.1%}</td>'
        f'<td style="padding:6px 12px;text-align:center">{avg_p3:.1%}</td>'
        f'<td style="padding:6px 12px;text-align:center">{avg_p2:.1%}</td></tr>'
    )
    display(HTML(
        f'<h3 style="font-family:monospace;margin-top:24px">Summary — {n} puzzle{"s" if n!=1 else ""}</h3>'
        f'<table style="border-collapse:collapse;font-family:monospace;font-size:0.9em">'
        f'{header}{rows}{avg_row}</table>'
    ))

    return per_puzzle_scores, dict(gcr=avg_gcr, acc=avg_acc, p3=avg_p3, p2=avg_p2)



In [6]:
import nltk
#nltk.download('wordnet')
from nltk.corpus import wordnet as wn


# {
#     "date": "2024/06/03",
#     "contest": "NYT Connections 358 - June 3rd, 2024",
#     "words": [
#         "LASER",
#         "PLUCK",
#         "THREAD",
#         "WAX",
#         "COIL",
#         "SPOOL",
#         "WIND",
#         "WRAP",
#         "HONEYCOMB",
#         "ORGANISM",
#         "SOLAR PANEL",
#         "SPREADSHEET",
#         "BALL",
#         "MOVIE",
#         "SCHOOL",
#         "VITAMIN"
#     ],
#     "answers": [
#         {
#             "answerDescription": "REMOVE, AS BODY HAIR",
#             "words": [
#                 "LASER",
#                 "PLUCK",
#                 "THREAD",
#                 "WAX"
#             ]
#         },
#         {
#             "answerDescription": "TWIST AROUND",
#             "words": [
#                 "COIL",
#                 "SPOOL",
#                 "WIND",
#                 "WRAP"
#             ]
#         },
#         {
#             "answerDescription": "THINGS MADE OF CELLS",
#             "words": [
#                 "HONEYCOMB",
#                 "ORGANISM",
#                 "SOLAR PANEL",
#                 "SPREADSHEET"
#             ]
#         },
#         {
#             "answerDescription": "B-___",
#             "words": [
#                 "BALL",
#                 "MOVIE",
#                 "SCHOOL",
#                 "VITAMIN"
#             ]
#         }
#     ],
#     "difficulty": 3.3
# },

laser = wn.synset('laser.n.01')

In [7]:
# "LASER",
# "PLUCK",
# "THREAD",
# "WAX"

# print(wn.synsets('laser')[0].definition())
# len(wn.synsets('laser')[0].lemmas())

# print(sorted(laser.hypernyms()))

pluck = wn.synset('pluck.n.01')
thread = wn.synset('thread.n.01')
wax = wn.synset('wax.n.01')

lpluck = laser.path_similarity(pluck)
lthread = laser.path_similarity(thread)
lwax = laser.path_similarity(wax)

print("laser-pluck:", lpluck)
print("laser-thread:", lthread)
print("laser-wax:", lwax)

pthread = pluck.path_similarity(thread)
pwax = pluck.path_similarity(wax)

print("pluck-thread:", pthread)
print("pluck-wax:", pwax)

twax = thread.path_similarity(wax)

print("thread-wax:", twax)



from nltk.corpus import wordnet_ic
nltk.download('wordnet_ic')
brown_ic = wordnet_ic.ic('ic-brown.dat')
semcor_ic = wordnet_ic.ic('ic-semcor.dat')

lpluck_ic = laser.res_similarity(pluck, brown_ic)
lthread_ic = laser.res_similarity(thread, brown_ic)
lwax_ic = laser.res_similarity(wax, brown_ic)
print("\nResnik similarity (using Brown IC):")
print("laser-pluck (IC):", lpluck_ic)
print("laser-thread (IC):", lthread_ic)
print("laser-wax (IC):", lwax_ic)

lpluck_ic = laser.res_similarity(pluck, semcor_ic)
lthread_ic = laser.res_similarity(thread, semcor_ic)
lwax_ic = laser.res_similarity(wax, semcor_ic)
print("\nResnik similarity (using Semcor IC):")
print("laser-pluck (IC):", lpluck_ic)
print("laser-thread (IC):", lthread_ic)
print("laser-wax (IC):", lwax_ic)

lpluck_ic = laser.lin_similarity(pluck, brown_ic)
lthread_ic = laser.lin_similarity(thread, brown_ic)
lwax_ic = laser.lin_similarity(wax, brown_ic)
print("\nLin similarity (using Brown IC):")
print("laser-pluck (IC):", lpluck_ic)
print("laser-thread (IC):", lthread_ic)
print("laser-wax (IC):", lwax_ic)

lpluck_ic = laser.lin_similarity(pluck, semcor_ic)
lthread_ic = laser.lin_similarity(thread, semcor_ic)
lwax_ic = laser.lin_similarity(wax, semcor_ic)
print("\nLin similarity (using Semcor IC):")
print("laser-pluck (IC):", lpluck_ic)
print("laser-thread (IC):", lthread_ic)
print("laser-wax (IC):", lwax_ic)

lpluck_ic = laser.wup_similarity(pluck, brown_ic)
lthread_ic = laser.wup_similarity(thread, brown_ic)
lwax_ic = laser.wup_similarity(wax, brown_ic)
print("\nWu-Palmer similarity (using Brown IC):")
print("laser-pluck (IC):", lpluck_ic)
print("laser-thread (IC):", lthread_ic)
print("laser-wax (IC):", lwax_ic)

lpluck_ic = laser.wup_similarity(pluck, semcor_ic)
lthread_ic = laser.wup_similarity(thread, semcor_ic)
lwax_ic = laser.wup_similarity(wax, semcor_ic)
print("\nWu-Palmer similarity (using Semcor IC):")
print("laser-pluck (IC):", lpluck_ic)
print("laser-thread (IC):", lthread_ic)
print("laser-wax (IC):", lwax_ic)

lpluck_ic = laser.jcn_similarity(pluck, brown_ic)
lthread_ic = laser.jcn_similarity(thread, brown_ic)
lwax_ic = laser.jcn_similarity(wax, brown_ic)
print("\nJiang-Conrath similarity (using Brown IC):")
print("laser-pluck (IC):", lpluck_ic)
print("laser-thread (IC):", lthread_ic)
print("laser-wax (IC):", lwax_ic)

lpluck_ic = laser.jcn_similarity(pluck, semcor_ic)
lthread_ic = laser.jcn_similarity(thread, semcor_ic)
lwax_ic = laser.jcn_similarity(wax, semcor_ic)
print("\nJiang-Conrath similarity (using Semcor IC):")
print("laser-pluck (IC):", lpluck_ic)
print("laser-thread (IC):", lthread_ic)
print("laser-wax (IC):", lwax_ic)

laser-pluck: 0.058823529411764705
laser-thread: 0.125
laser-wax: 0.07142857142857142
pluck-thread: 0.0625
pluck-wax: 0.0625
thread-wax: 0.07692307692307693

Resnik similarity (using Brown IC):
laser-pluck (IC): -0.0
laser-thread (IC): 2.305848731451232
laser-wax (IC): 0.8017591149538994

Resnik similarity (using Semcor IC):
laser-pluck (IC): -0.0
laser-thread (IC): 2.493384085748939
laser-wax (IC): 0.6143639493869085

Lin similarity (using Brown IC):
laser-pluck (IC): -0.0
laser-thread (IC): 4.611697462902464e-300
laser-wax (IC): 1.6035182299077988e-300

Lin similarity (using Semcor IC):
laser-pluck (IC): -0.0
laser-thread (IC): 4.9867681714978775e-300
laser-wax (IC): 1.228727898773817e-300

Wu-Palmer similarity (using Brown IC):
laser-pluck (IC): 0.1111111111111111
laser-thread (IC): 0.5882352941176471
laser-wax (IC): 0.23529411764705882

Wu-Palmer similarity (using Semcor IC):
laser-pluck (IC): 0.1111111111111111
laser-thread (IC): 0.5882352941176471
laser-wax (IC): 0.235294117647058

[nltk_data] Downloading package wordnet_ic to /Users/yt/nltk_data...
[nltk_data]   Package wordnet_ic is already up-to-date!


In [8]:


# Function for GPT to call WordNet

def analyze_wordnet_similarity(words):
    """Return pairwise WordNet similarity scores for the provided words."""
    similarities = {}
    for i, word1 in enumerate(words):
        for j, word2 in enumerate(words):
            if i < j:
                try:
                    syn1 = wn.synsets(word1.lower())[0] if wn.synsets(word1.lower()) else None
                    syn2 = wn.synsets(word2.lower())[0] if wn.synsets(word2.lower()) else None
                    best_score = 0
                    if syn1 and syn2:
                        candidates = []
                        try:
                            candidates.append(syn1.path_similarity(syn2))
                        except Exception:
                            pass

                        for ic in (globals().get('brown_ic'), globals().get('semcor_ic')):
                            if ic is not None:
                                try:
                                    candidates.append(syn1.res_similarity(syn2, ic))
                                except Exception:
                                    pass
                                try:
                                    candidates.append(syn1.lin_similarity(syn2, ic))
                                except Exception:
                                    pass
                                try:
                                    candidates.append(syn1.jcn_similarity(syn2, ic))
                                except Exception:
                                    pass

                        try:
                            candidates.append(syn1.wup_similarity(syn2))
                        except Exception:
                            pass

                        valid_scores = [score for score in candidates if score is not None]
                        if valid_scores:
                            best_score = max(valid_scores)
                    similarities[f"{word1}-{word2}"] = round(best_score, 3)
                except:
                    similarities[f"{word1}-{word2}"] = 0
    return similarities


def analyze_wordnet_hypernyms(words):
    """Return the top hypernyms for each word."""
    hypernyms = {}
    for word in words:
        try:
            syn = wn.synsets(word.lower())[0] if wn.synsets(word.lower()) else None
            if syn:
                hypernyms[word] = [h.name().split('.')[0] for h in syn.hypernyms()[:3]]
            else:
                hypernyms[word] = []
        except:
            hypernyms[word] = []
    return hypernyms


def analyze_wordnet_definitions(words):
    """Return the primary WordNet definition for each word."""
    definitions = {}
    for word in words:
        try:
            syn = wn.synsets(word.lower())[0] if wn.synsets(word.lower()) else None
            if syn:
                definitions[word] = syn.definition()
            else:
                definitions[word] = "No definition found"
        except:
            definitions[word] = "Error getting definition"
    return definitions


def analyze_wordnet_relationships(words):
    """
    Analyze relationships between words using WordNet.

    Args:
        words: List of words to analyze

    Returns:
        Dictionary with all available analysis results.
    """
    return {
        "similarity": analyze_wordnet_similarity(words),
        "hypernyms": analyze_wordnet_hypernyms(words),
        "definitions": analyze_wordnet_definitions(words),
    }

# NYT Connections game data
game_data = {
    "words": [
        "LASER", "PLUCK", "THREAD", "WAX",
        "COIL", "SPOOL", "WIND", "WRAP",
        "HONEYCOMB", "ORGANISM", "SOLAR PANEL", "SPREADSHEET",
        "BALL", "MOVIE", "SCHOOL", "VITAMIN"
    ]
}

In [42]:
import ipywidgets as widgets
from IPython.display import display, Markdown

import os
import json
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI(
     base_url= _URL,
     api_key=os.getenv(_GitHUB_TOKEN)
   # base_url="https://models.inference.ai.azure.com",
   # api_key=os.getenv("GITHUB_TOKEN")
)



# Function definitions for GPT
tools = [
    {
        "type": "function",
        "function": {
            "name": "analyze_wordnet_relationships",
            "description": "Analyze semantic relationships between words using WordNet",
            "parameters": {
                "type": "object",
                "properties": {
                    "words": {
                        "type": "array",
                        "items": {"type": "string"},
                        "description": "List of words to analyze"
                    }
                },
                "required": ["words"]
            }
        }
    }
]

def ask_gpt(messages, prompt, use_tools):
    # Use a fresh local conversation each call, keeping only the system prompt
    local_messages = [messages[0], {"role": "user", "content": prompt}]

    kwargs = {}
    if use_tools:
        kwargs["tools"] = tools
        kwargs["tool_choice"] = "auto"

    completion = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=local_messages,
        **kwargs
    )

    if completion.choices and len(completion.choices) > 0:
        message = completion.choices[0].message
        tool_calls = getattr(message, "tool_calls", None)
        print("Tools:", getattr(message, "tool_calls", ""))
        if tool_calls:
            # Append the assistant's reply (with tool_calls) to the conversation
            local_messages.append(message)

            for tool_call in tool_calls:
                tool_name = tool_call.function.name
                tool_args = json.loads(tool_call.function.arguments)

                if tool_name == "analyze_wordnet_relationships":
                    tool_result = analyze_wordnet_relationships(**tool_args)
                else:
                    tool_result = {"error": f"Unknown tool: {tool_name}"}

                # tool_call_id is required to match this result to the right call
                local_messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": json.dumps(tool_result)
                })

            followup = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=local_messages
            )
            if followup.choices:
                return followup.choices[0].message.content

        return getattr(message, "content", None)
    return str(completion)




In [43]:
# Select Words:

import json
import random

with open(_NYT_CONNECTION_FILE) as f:
    dataset = json.load(f)

##Filters
DIFFICULTY            = _DIFFICULTY             # Set to None or "" to skip difficulty filter
GAME_DATE             = _GAME_DATE              # Set to None or "" to skip date filter (format: "YYYY/MM/DD")
NUMBER_OF_CONNECTIONS = _NUMBER_OF_CONNECTIONS  # Number of puzzles to randomly sample; set to None or 0 to use all

pool = dataset
if DIFFICULTY:
    pool = [p for p in pool if p["difficulty"] == DIFFICULTY]
if GAME_DATE:
    pool = [p for p in pool if p["date"] == GAME_DATE]
if NUMBER_OF_CONNECTIONS:
    pool = random.sample(pool, min(NUMBER_OF_CONNECTIONS, len(pool)))

if not pool:
    raise ValueError(f"No puzzles found for DIFFICULTY={DIFFICULTY!r}, GAME_DATE={GAME_DATE!r}")


print(f"Selected {len(pool)} puzzle(s) of difficulty {DIFFICULTY} and date '{GAME_DATE}' for evaluation.")

#Baseline: no hints provied to GPT
raw_replies_without_hints = []
all_actual_without_hints = []
for i, puzzle in enumerate(pool):
    #message building for GPT prompt
    # System prompt for NYT Connections game
    messages = [
            {"role": "system", "content": f"""You are an expert at solving NYT Connections puzzles. 

        The puzzle has 16 words in 4 groups of 4. Each group shares a common theme. Analyze the relationships to find the groups.

        Current puzzle words: {", ".join(puzzle["words"])}

        Think step by step and use your knowledge to find the groups. 
        Return ONLY a JSON array with exactly 4 objects, each with:
        - "answerDescription": a short ALL-CAPS description of the category
        - "words": a list of exactly 4 words from the input

        Example format:
        [
          {{"answerDescription": "CATEGORY NAME", "words": ["WORD1", "WORD2", "WORD3", "WORD4"]}},
          ...
        ]
        """}
        ]  

    # Test GPT with Baseline (no hints)
    print("\nTesting GPT with Baseline (no hints) grouping...")
    group_prompt = (
        "Group the following 16 words into exactly 4 groups of 4 words each, " + ", ".join(puzzle["words"]) + ". "
        + "and return only the groups as lists: " )
    
    raw_reply = ask_gpt(messages, group_prompt, use_tools=False)
    raw_replies_without_hints.append(raw_reply)
    all_actual_without_hints.append(puzzle["answers"])
    print(f"  [{i+1}/{len(pool)}] done: {puzzle['contest']}")


#----------------With Wordnet------------------------------
raw_replies_with_hints = []
all_actual_with_hints = []
for i, puzzle in enumerate(pool):
    #message building for GPT prompt
    # System prompt for NYT Connections game
    messages = [
            {"role": "system", "content": f"""You are an expert at solving NYT Connections puzzles. You have access to WordNet to analyze semantic relationships between words.

        Use the analyze_wordnet_relationships function to:
        - Find similarities between words
        - Get hypernyms (broader categories) for words
        - Get definitions of words
        The function returns all supported analyses at once.

        The puzzle has 16 words in 4 groups of 4. Each group shares a common theme. Analyze the relationships to find the groups.

        Current puzzle words: {", ".join(puzzle["words"])}

        Think step by step and use WordNet data to support your reasoning.
        Return ONLY a JSON array with exactly 4 objects, each with:
        - "answerDescription": a short ALL-CAPS description of the category
        - "words": a list of exactly 4 words from the input

        Example format:
        [
          {{"answerDescription": "CATEGORY NAME", "words": ["WORD1", "WORD2", "WORD3", "WORD4"]}},
          ...
        ]
        """}
        ]
    
    
    
    #------------------------------------------
    # Test the WordNet integration
    result = analyze_wordnet_relationships(puzzle["words"])
    print("Similarities:", result["similarity"])
    print("Definitions:", result["definitions"])
    print("Hypernyms:", result["hypernyms"])

    # Test GPT with WordNet
    print("\nTesting GPT with WordNet grouping...")
    group_prompt = (
        "Group the following 16 words into exactly 4 groups of 4 words each, " + ", ".join(puzzle["words"]) + ". "
        + "and return only the groups as lists: " )
    
    raw_reply = ask_gpt(messages, group_prompt, use_tools=True)
    raw_replies_with_hints.append(raw_reply)
    all_actual_with_hints.append(puzzle["answers"])
    print(f"  [{i+1}/{len(pool)}] done: {puzzle['contest']}")


Selected 4 puzzle(s) of difficulty 4 and date '' for evaluation.

Testing GPT with Baseline (no hints) grouping...
Tools: None
  [1/4] done: NYT Connections 135 - October 24th, 2023

Testing GPT with Baseline (no hints) grouping...
Tools: None
  [2/4] done: NYT Connections 172 - November 30th, 2023

Testing GPT with Baseline (no hints) grouping...
Tools: None
  [3/4] done: NYT Connections 314 - April 20th, 2024

Testing GPT with Baseline (no hints) grouping...
Tools: None
  [4/4] done: NYT Connections 304 - April 10th, 2024
Similarities: {'COLONY-HERD': 2.872, 'COLONY-PRIDE': 0.779, 'COLONY-SCHOOL': 3.527, 'COLONY-CRANNY': 0.779, 'COLONY-NICHE': 0.779, 'COLONY-NOOK': 0.143, 'COLONY-RECESS': 0.779, 'COLONY-CLASSIC': 0.154, 'COLONY-DEFINITIVE': 0.222, 'COLONY-MODEL': 0.779, 'COLONY-TEXTBOOK': 0.118, 'COLONY-BACKPACK': 0.133, 'COLONY-BIGWIG': 0.154, 'COLONY-DOWNTOWN': 0.143, 'COLONY-RAGTAG': 2.872, 'HERD-PRIDE': 0.779, 'HERD-SCHOOL': 2.872, 'HERD-CRANNY': 0.779, 'HERD-NICHE': 0.779, 'HERD

In [44]:
## Show Results for Baseline (no hints):
scores_no_hints, summary_no_hints = evaluate_run(
    raw_replies_without_hints,
    all_actual_without_hints,
)

Predicted,Actual
"✓ ANIMALSHERD, PRIDE, SCHOOL, COLONY","ANIMAL GROUPSCOLONY, HERD, PRIDE, SCHOOL"
"✓ HIDDEN PLACESCRANNY, NICHE, NOOK, RECESS","SMALL OPENINGCRANNY, NICHE, NOOK, RECESS"
"✓ EDUCATIONAL TERMSCLASSIC, DEFINITIVE, MODEL, TEXTBOOK","PARADIGMATICCLASSIC, DEFINITIVE, MODEL, TEXTBOOK"
"✓ SOCIAL STATUSBACKPACK, BIGWIG, DOWNTOWN, RAGTAG","RHYMING COMPOUND WORDSBACKPACK, BIGWIG, DOWNTOWN, RAGTAG"


Predicted,Actual
"✓ WORDS THAT MEAN TO AVOIDDODGE, DUCK, ESCAPE, SKIRT","AVOIDDODGE, DUCK, ESCAPE, SKIRT"
"✓ BIRDSGOOSE, ROBIN, WATSON, HOBBES","HITCHCOCK MOVIESBIRDS, NOTORIOUS, REBECCA, ROPE"
"✗ LITERATUREREBECCA, NOTORIOUS, COTTAGE, STRINGbest match: HITCHCOCK MOVIES (2/4)","SIDEKICKSGOOSE, HOBBES, ROBIN, WATSON"
"✗ CREAM AND RELATEDCREAM, BIRDS, SAY, ROPEbest match: HITCHCOCK MOVIES (2/4)","___ CHEESECOTTAGE, CREAM, SAY, STRING"


Predicted,Actual
"✓ FALSEHOODSBUNK, CROCK, HOGWASH, HORSEFEATHERS","BALDERDASHBUNK, CROCK, HOGWASH, HORSEFEATHERS"
"✗ TOOL OR IMPLEMENTHAMMER, BATON, PITCHFORK, HURDLEbest match: TRACK AND FIELD EQUIPMENT (3/4)","TRACK AND FIELD EQUIPMENTBATON, HAMMER, HURDLE, POLE"
"✓ KNOTSBOWLINE, HITCH, SHEEPSHANK, BEND","PARTS OF A DEVIL COSTUMEGOATEE, HORNS, PITCHFORK, TAIL"
"✗ ANIMALSGOATEE, HORNS, TAIL, POLEbest match: PARTS OF A DEVIL COSTUME (3/4)","TYPES OF KNOTSBEND, BOWLINE, HITCH, SHEEPSHANK"


Predicted,Actual
"✓ LEADERSHIP ROLESCHAIR, CHIEF, DIRECTOR, HEAD","PERSON IN CHARGECHAIR, CHIEF, DIRECTOR, HEAD"
"✓ TYPES OF GROUNDFIELD, GREEN, GROUNDS, LAWN","GRASSY AREAFIELD, GREEN, GROUNDS, LAWN"
"✓ NUTSCHEST, COCO, HAZEL, PEA","WORDS BEFORE “NUT”CHEST, COCO, HAZEL, PEA"
"✓ COLOR NAMESBROWN, DOGS, FICTION, UNCHAINED","SECOND WORDS IN TARANTINO MOVIESBROWN, DOGS, FICTION, UNCHAINED"


Contest,Solved,Group Acc,Partial-3,Partial-2
"NYT Connections 135 - October 24th, 2023",✓,100%,100%,100%
"NYT Connections 172 - November 30th, 2023",✗,50%,50%,100%
"NYT Connections 314 - April 20th, 2024",✗,50%,100%,100%
"NYT Connections 304 - April 10th, 2024",✓,100%,100%,100%
Average (4),50.0%,75.0%,87.5%,100.0%


In [45]:
## Show Results for WordNet:
scores_with_hints, summary_with_hints = evaluate_run(
    raw_replies_with_hints,
    all_actual_with_hints
)

Predicted,Actual
"✓ GROUP OF ANIMALSCOLONY, HERD, PRIDE, SCHOOL","ANIMAL GROUPSCOLONY, HERD, PRIDE, SCHOOL"
"✓ TIGHT SPACESCRANNY, NICHE, NOOK, RECESS","SMALL OPENINGCRANNY, NICHE, NOOK, RECESS"
"✓ EDUCATIONAL MATERIALCLASSIC, DEFINITIVE, MODEL, TEXTBOOK","PARADIGMATICCLASSIC, DEFINITIVE, MODEL, TEXTBOOK"
"✓ URBAN TERMSBACKPACK, BIGWIG, DOWNTOWN, RAGTAG","RHYMING COMPOUND WORDSBACKPACK, BIGWIG, DOWNTOWN, RAGTAG"


Predicted,Actual
"✓ ACTIONS TO AVOIDDODGE, DUCK, ESCAPE, SKIRT","AVOIDDODGE, DUCK, ESCAPE, SKIRT"
"✗ BIRDSBIRDS, GOOSE, ROBIN, HOBBESbest match: SIDEKICKS (3/4)","HITCHCOCK MOVIESBIRDS, NOTORIOUS, REBECCA, ROPE"
"✗ FAMOUS NAMESNOTORIOUS, REBECCA, WATSON, HOBBESbest match: HITCHCOCK MOVIES (2/4)","SIDEKICKSGOOSE, HOBBES, ROBIN, WATSON"
"✓ HOUSEHOLD ITEMSCOTTAGE, CREAM, SAY, STRING","___ CHEESECOTTAGE, CREAM, SAY, STRING"


Predicted,Actual
"✓ USELESS STATEMENTSBUNK, CROCK, HOGWASH, HORSEFEATHERS","BALDERDASHBUNK, CROCK, HOGWASH, HORSEFEATHERS"
"✓ SPORTS EQUIPMENTBATON, HAMMER, HURDLE, POLE","TRACK AND FIELD EQUIPMENTBATON, HAMMER, HURDLE, POLE"
"✓ FARM TOOLS/ANIMALSGOATEE, HORNS, PITCHFORK, TAIL","PARTS OF A DEVIL COSTUMEGOATEE, HORNS, PITCHFORK, TAIL"
"✓ KNOTSBEND, BOWLINE, HITCH, SHEEPSHANK","TYPES OF KNOTSBEND, BOWLINE, HITCH, SHEEPSHANK"


Predicted,Actual
"✓ LEADERSHIP ROLESCHAIR, CHIEF, DIRECTOR, HEAD","PERSON IN CHARGECHAIR, CHIEF, DIRECTOR, HEAD"
"✓ LAWN & GARDENFIELD, GREEN, GROUNDS, LAWN","GRASSY AREAFIELD, GREEN, GROUNDS, LAWN"
"✓ NUTS & LEGUMESCHEST, COCO, HAZEL, PEA","WORDS BEFORE “NUT”CHEST, COCO, HAZEL, PEA"
"✓ GENRES OF FICTIONBROWN, DOGS, FICTION, UNCHAINED","SECOND WORDS IN TARANTINO MOVIESBROWN, DOGS, FICTION, UNCHAINED"


Contest,Solved,Group Acc,Partial-3,Partial-2
"NYT Connections 135 - October 24th, 2023",✓,100%,100%,100%
"NYT Connections 172 - November 30th, 2023",✗,50%,75%,100%
"NYT Connections 314 - April 20th, 2024",✓,100%,100%,100%
"NYT Connections 304 - April 10th, 2024",✓,100%,100%,100%
Average (4),75.0%,87.5%,93.8%,100.0%


In [46]:
## Comparison: No Hints vs With Hints
from IPython.display import display, HTML

metrics = ["gcr", "acc", "p3", "p2"]
labels  = ["Game Completion Rate", "Group Accuracy", "Partial-3 Rate", "Partial-2 Rate"]

def _delta_cell(a, b):
    diff = b - a
    color = "#66bb6a" if diff > 0 else ("#ef5350" if diff < 0 else "#aaa")
    arrow = "▲" if diff > 0 else ("▼" if diff < 0 else "—")
    return (f'<td style="padding:6px 14px;text-align:center;color:{color};font-weight:bold">'
            f'{arrow} {abs(diff):.1%}</td>')

header = (
    '<tr style="color:#aaa;border-bottom:1px solid #555">'
    '<th style="padding:6px 14px;text-align:left">Metric</th>'
    '<th style="padding:6px 14px">No Hints (A)</th>'
    '<th style="padding:6px 14px">With Hints (B)</th>'
    '<th style="padding:6px 14px">Δ (B − A)</th>'
    '</tr>'
)
rows = "".join(
    f'<tr>'
    f'<td style="padding:6px 14px">{lbl}</td>'
    f'<td style="padding:6px 14px;text-align:center">{summary_no_hints[m]:.1%}</td>'
    f'<td style="padding:6px 14px;text-align:center">{summary_with_hints[m]:.1%}</td>'
    f'{_delta_cell(summary_no_hints[m], summary_with_hints[m])}'
    f'</tr>'
    for m, lbl in zip(metrics, labels)
)

display(HTML(
    '<h2 style="font-family:monospace;border-bottom:2px solid #555;padding-bottom:6px">'
    'Comparison: No Hints vs With WordNet Hints</h2>'
    '<table style="border-collapse:collapse;font-family:monospace;font-size:0.95em">'
    f'{header}{rows}</table>'
    '<p style="color:#888;font-size:0.85em;font-family:monospace">'
    '▲ = With-hints better &nbsp;|&nbsp; ▼ = No-hints better</p>'
))


Metric,No Hints (A),With Hints (B),Δ (B − A)
Game Completion Rate,50.0%,75.0%,▲ 25.0%
Group Accuracy,75.0%,87.5%,▲ 12.5%
Partial-3 Rate,87.5%,93.8%,▲ 6.2%
Partial-2 Rate,100.0%,100.0%,— 0.0%
